In [2]:
from neo4j import GraphDatabase
from google import genai
import os

In [3]:
main_driver = GraphDatabase.driver(uri="neo4j://127.0.0.1:7687",auth=("neo4j","Goppanmavane@2"))

client = genai.Client(api_key="<your_gemini_key>")
MODEL = "gemini-2.5-flash"

In [20]:
def generate_cypher(query):
    prompt = f"""
You are an expert in Neo4j Cypher.

Graph Schema:

Nodes:
- (FoodnHerb {{name, type, scientific_name, category, description}})
- (Nutrients {{name}})
- (Property {{name}})
- (Symptom {{name}})
- (BodySystem {{name}})
- (Compound {{name}})

Relationships:
- (FoodnHerb)-[:HAS_PROPERTY]->(Property)
- (FoodnHerb)-[:HELPS_WITH]->(Symptom)
- (FoodnHerb)-[:CONTAINS]->(Compound)
- (FoodnHerb)-[:COMBINES_WITH]->(FoodnHerb)

- (Nutrients)-[:FOUND_IN]->(FoodnHerb)
- (Nutrients)-[:SUPPORTS_SYSTEM]->(BodySystem)
- (Nutrients)-[:DEFICIENCY_CAUSES]->(Symptom)
- (Nutrients)-[:INTERACTS_WITH]-(Nutrients)   // undirected

Important Rules:
- Only return Cypher query (no explanation, no markdown)
- Use CONTAINS for fuzzy matching (e.g., s.name CONTAINS "cough")
- Always use correct relationship directions except INTERACTS_WITH which is undirected
- Limit results to 20
- Avoid returning duplicate nodes
- Use appropriate labels based on query intent:
    - "herb" → FoodnHerb with type = "herb"
    - "food" → FoodnHerb with type = "food"
- Return meaningful fields like names (e.g., h.name, n.name)

Examples:

User: herbs for cough  
Cypher:
MATCH (h:FoodnHerb)-[:HELPS_WITH]->(s:Symptom)
WHERE h.type = "herb" AND s.name CONTAINS "cough"
RETURN h.name
LIMIT 20

User: nutrients for immune system  
Cypher:
MATCH (n:Nutrients)-[:SUPPORTS_SYSTEM]->(b:BodySystem)
WHERE b.name CONTAINS "immune"
RETURN n.name
LIMIT 20

User Query:
{query}
"""
    
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    
    return response.text.strip()

In [5]:
def run_cypher(cypher):
    with main_driver.session() as session:
        result = session.run(cypher)
        return [record.data() for record in result]

In [ ]:
def test_query(user_query):
    print("\n🔍 User Query:", user_query)

    cypher = generate_cypher(user_query)
    # print("\n⚡ Generated Cypher:\n", cypher)

    try:
        results = run_cypher(cypher)
        print("\n📊 Results:")
        for r in results:
            print(r)
    except Exception as e:
        print("❌ Error executing query:", e)

In [29]:
test_query("Which herbs help with Inflammation")


🔍 User Query: Which herbs help with Inflammation

⚡ Generated Cypher:
 MATCH (h:FoodnHerb)-[:HELPS_WITH]->(s:Symptom)
WHERE h.type = "herb" AND s.name CONTAINS "Inflammation"
RETURN h.name
LIMIT 20

📊 Results:
{'h.name': 'Black Pepper'}
{'h.name': 'Ginger'}
{'h.name': 'Honey'}
{'h.name': 'Garlic'}
{'h.name': 'Neem'}
{'h.name': 'Aloe Vera'}
